In [ ]:
import pandas as pd
import plotly.io as pio
pio.templates.default = "plotly_white"
import xarray as xr
from whakaaribn.visualize import trellis_plot
from whakaaribn import get_color

In [ ]:
try:
    data_file = snakemake.input.data
    forecast_all_data = snakemake.input.forecast_all_data
    forecast_uncertainty = snakemake.input.forecast_uncertainty
except NameError as e:
    data_file = '../data/whakaari_data_with_groups.csv'
    forecast_all_data = '../forecasts/whakaari_forecasts.nc'
    forecast_uncertainty = '../forecasts/whakaari_uncertainty.nc'

In [ ]:
xds_best = xr.open_dataset(forecast_all_data)
data = pd.read_csv(data_file, parse_dates=True, index_col=0)

In [ ]:
xds_all_1 = xr.open_dataset(forecast_uncertainty)

In [ ]:
median_model = xds_all_1.median('model_score').to_array()
# min_model = xds_all_1.chunk(dict(model_score=-1)).quantile(0.15, 'model_score')
# max_model = xds_all_1.chunk(dict(model_score=-1)).quantile(0.85, 'model_score')
min_model = xds_all_1.chunk(dict(model_score=-1)).min('model_score').to_array()
max_model = xds_all_1.chunk(dict(model_score=-1)).max('model_score').to_array()
models = {"Eruption Probability (median model)": {'model': median_model.squeeze('variable'), 'color': get_color(0)},
          "Eruption Probability (best model)": {'model': xds_best['probs'], 'color': get_color(1)},
          "min": {'model': min_model.squeeze('variable'), 'color': get_color(0, alpha=0.3)},
          "max": {'model': max_model.squeeze('variable'), 'color': get_color(0, alpha=0.3)}
}
fig = trellis_plot(models, data, plot_uncertainty='quantile')
try:
    fig.write_image(snakemake.output.forecast_uncertainty_plot, width=1200, height=1000, scale=5)
except NameError:
    pass
fig